## Bronze Layer - ERCOT Wind & Solar Generation

Read raw ERCOT hourly wind and solar generation data from the source table, adds ingestion metadata (timestamp and source) renames columns to clean snake_case format, and writes to the medallion lakehouse Bronze layer as a Delta table.

**Source:** dbacademy.default.ercot_solar_2023 (2023-2025 data, ~52k rows). 

**Target:** workspace.medallion_lakehouse.bronze_generation. 

**Layer:** Bronze - raw data preserved, no business logic applied. 

In [0]:
# this creates a project schema (like creating a dedicated project folder)
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.medallion_lakehouse")
print("Schema created successfully")

In [0]:
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql import functions as F

In [0]:
raw_df = spark.table("dbacademy.default.ercot_solar_2023")
raw_df.show(5)

In [0]:
# These tell us when this data was loaded and where it came from
bronze_df = raw_df\
    .withColumn("ingest_timestamp", current_timestamp())\
        .withColumn("source_table", lit("dbacademy.default.ercot_solar_2023"))

In [0]:
# show the shape to verify
print(f"Row count: {bronze_df.count()}")
print(f"Column count: {len(bronze_df.columns)}")
bronze_df.show(3)

In [0]:
# Rename all columns to clean snake_case names
bronze_clean_df = bronze_df\
    .withColumnRenamed("Time (Hour-Ending)", "hour_ending")\
    .withColumnRenamed("Date", "date_label")\
    .withColumnRenamed("ERCOT.LOAD", "ercot_load_mw")\
    .withColumnRenamed("ERCOT.WIND.GEN", "wind_gen_mw")\
    .withColumnRenamed("Total Wind Installed, MW", "total_wind_installed_mw")\
    .withColumnRenamed("Wind Output, % of Load", "wind_pct_of_load")\
    .withColumnRenamed("Wind Output, % of Installed", "wind_capacity_factor")\
    .withColumnRenamed("Wind 1-hr MW change", "wind_1hr_mw_change")\
    .withColumnRenamed("Wind 1-hr % change", "wind_1hr_pct_change")\
    .withColumnRenamed("ERCOT.PVGR.GEN", "solar_gen_mw")\
    .withColumnRenamed("Total Solar Installed, MW", "total_solar_installed_mw")\
    .withColumnRenamed("Solar Output, % of Load", "solar_pct_of_load")\
    .withColumnRenamed("Solar Output, % of Installed", "solar_capacity_factor")\
    .withColumnRenamed("Solar 1-hr MW change", "solar_1hr_mw_change")\
    .withColumnRenamed("Solar 1-hr % change", "solar_1hr_pct_change")\
    .withColumnRenamed("Daytime Hour", "daytime_hour")\
    .withColumnRenamed("Ramping Daytime Hour", "ramping_daytime_hour")

#confirm new column names
print(bronze_clean_df.columns)

Delta adds version history, ACID transactions and time travel on tops Parquet files underneath

In [0]:
# Write as Delta table to Bronze Layer
bronze_clean_df.write\
    .format("delta")\
        .mode("overwrite")\
            .saveAsTable("workspace.medallion_lakehouse.bronze_generation")

print("Bronze layer written successfully")

In [0]:
#Final Bronze verification
spark.sql("""
          select
            count(*) as total_records,
            min(hour_ending) as earliest_record,
            max(hour_ending) as latest_record,
            sum(case when wind_gen_mw is null then 1 else 0 end) as wind_nulls,
            sum(case when solar_gen_mw is null then 1 else 0 end) as solar_nulls 
          from workspace.medallion_lakehouse.bronze_generation
          """).show(truncate=False)